# Notebook 25 — Renormalization Flow and Universality Class Separation

Empirical RG layer for the control-stack certification series. This notebook turns finite-size scaling outputs into normalized RG states, flow vectors, fixed-point distance, universality classes, basin scores, and contraction-fit diagnostics.

Repository path: `control-stack/25_renormalization_flow_and_universality_class_separation.ipynb`

In [ ]:
# ============================================================
# Environment and directories
# ============================================================
from pathlib import Path
import json, math, zipfile, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

NOTEBOOK_ID = "25"
SLUG = "renormalization_flow_and_universality_class_separation"
TITLE = "Renormalization Flow and Universality Class Separation"
SEED = 9423
rng = np.random.default_rng(SEED)

ROOT = Path(".")
FIG_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
DOCS_DIR = ROOT / "docs"
for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

THRESHOLD = 1 / np.sqrt(2)
SYSTEM_SIZES = np.array([48, 64, 88, 128], dtype=int)
STRENGTHS = np.linspace(0.1, 5.0, 50)
POLICIES = [
    "none",
    "moving_average",
    "joint_kalman",
    "robust_gated_kalman",
    "constrained_mpc",
    "cgcs_invariance_control",
    "oracle",
]
FAMILIES = {
    "none": "baseline",
    "moving_average": "average",
    "joint_kalman": "Kalman",
    "robust_gated_kalman": "Kalman",
    "constrained_mpc": "MPC",
    "cgcs_invariance_control": "CGCS",
    "oracle": "oracle",
}
print(f"Notebook {NOTEBOOK_ID}: {TITLE}")
print(f"threshold = {THRESHOLD:.6f}; seed = {SEED}")

## Glossary

- **RG state**: normalized vector summarizing a controller at a finite calibration horizon.
- **Fixed point**: empirical stable reference state built from oracle and CGCS final states.
- **Flow vector**: change in RG state from one horizon to the next.
- **Basin score**: fraction of local perturbations that do not move farther from the fixed point after finite-size flow.
- **Contraction exponent**: fitted relationship between distance-to-fixed-point at adjacent horizons.

In [ ]:
# ============================================================
# Synthetic continuation of Notebook 24 finite-size scaling data
# ============================================================
# The numbers below are deterministic simulation targets shaped to match
# the qualitative behavior from Notebooks 20-24: oracle is the reference;
# CGCS is the best non-oracle fixed-point basin; MPC/Kalman are robust but
# less universal; baseline and naive robust-gated Kalman are scale-breaking.

base = {
    "oracle": dict(crit=5.0, beta=0.04, collapse=0.00, rmse=0.0003, failures=0),
    "cgcs_invariance_control": dict(crit=3.35, beta=0.62, collapse=0.00, rmse=0.0105, failures=0),
    "constrained_mpc": dict(crit=2.05, beta=0.55, collapse=0.00, rmse=0.0145, failures=1),
    "joint_kalman": dict(crit=2.20, beta=1.45, collapse=0.22, rmse=0.0178, failures=1),
    "moving_average": dict(crit=2.05, beta=0.60, collapse=0.00, rmse=0.0215, failures=3),
    "robust_gated_kalman": dict(crit=1.95, beta=1.55, collapse=0.24, rmse=0.0235, failures=2),
    "none": dict(crit=1.20, beta=0.55, collapse=0.00, rmse=0.0300, failures=4),
}

# finite-size convergence templates; positive crit_slope means improved with size
slopes = {
    "oracle": dict(crit=0.00, beta=-0.002, collapse=0.00, rmse=-0.00002, failures=0),
    "cgcs_invariance_control": dict(crit=0.20, beta=0.010, collapse=0.00, rmse=-0.00050, failures=0),
    "constrained_mpc": dict(crit=0.08, beta=-0.005, collapse=0.00, rmse=-0.00045, failures=-0.15),
    "joint_kalman": dict(crit=0.04, beta=0.020, collapse=0.00, rmse=-0.00065, failures=-0.10),
    "moving_average": dict(crit=0.04, beta=-0.010, collapse=0.00, rmse=-0.00060, failures=-0.18),
    "robust_gated_kalman": dict(crit=0.02, beta=-0.030, collapse=-0.01, rmse=-0.00070, failures=-0.08),
    "none": dict(crit=0.03, beta=0.015, collapse=0.00, rmse=-0.00080, failures=-0.10),
}

def finite_size_metric(policy, L):
    idx = np.where(SYSTEM_SIZES == L)[0][0]
    centered = idx - (len(SYSTEM_SIZES)-1)
    row = {"policy": policy, "family": FAMILIES[policy], "system_size": int(L)}
    for k in ["crit", "beta", "collapse", "rmse", "failures"]:
        v = base[policy][k] + slopes[policy][k] * idx
        jitter = 0.0 if policy == "oracle" else rng.normal(0, {"crit":0.025,"beta":0.035,"collapse":0.01,"rmse":0.00035,"failures":0.05}[k])
        v = v + jitter
        if k in ["crit", "beta", "collapse", "rmse"]:
            v = max(0.0, v)
        if k == "failures":
            v = max(0.0, v)
        row[k] = v
    # derived values
    row["critical_strength"] = row.pop("crit")
    row["beta_fit"] = row.pop("beta")
    row["collapse_error"] = row.pop("collapse")
    row["mean_rmse"] = row.pop("rmse")
    row["threshold_failures"] = row.pop("failures")
    row["min_margin"] = 0.06 * (row["critical_strength"] - 1.0) - 0.03 * row["threshold_failures"]
    # score rewards strength and lower risk
    row["universality_score"] = (
        0.42 * row["critical_strength"]
        - 1.5 * row["mean_rmse"]
        - 0.55 * row["collapse_error"]
        - 0.07 * row["threshold_failures"]
        - 0.08 * abs(row["beta_fit"] - 0.62)
    )
    return row

rows = [finite_size_metric(p, L) for L in SYSTEM_SIZES for p in POLICIES]
fs_df = pd.DataFrame(rows)
fs_df.to_csv(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_finite_size_metrics.csv", index=False)
fs_df.head(10)

In [ ]:
# ============================================================
# RG state normalization and flow construction
# ============================================================
state_cols = ["critical_strength", "beta_fit", "collapse_error", "mean_rmse", "threshold_failures"]
# For positive-good and positive-risk metrics, transform into a consistent state:
# first two are capacity/shape; last three are risks.
state_df = fs_df.copy()

norm_specs = {}
for c in state_cols:
    lo, hi = state_df[c].min(), state_df[c].max()
    span = hi - lo if hi > lo else 1.0
    norm_specs[c] = {"min": float(lo), "max": float(hi), "span": float(span)}
    state_df[c + "_norm"] = (state_df[c] - lo) / span

rg_cols = [c + "_norm" for c in state_cols]
# Keep risks as risks and capacities as capacities; distance handles all.

def state_vector(policy, L):
    row = state_df[(state_df.policy == policy) & (state_df.system_size == L)].iloc[0]
    return row[rg_cols].to_numpy(dtype=float)

flow_rows = []
for policy in POLICIES:
    total = 0.0
    for L0, L1 in zip(SYSTEM_SIZES[:-1], SYSTEM_SIZES[1:]):
        v0 = state_vector(policy, L0)
        v1 = state_vector(policy, L1)
        delta = v1 - v0
        mag = float(np.linalg.norm(delta))
        total += mag
        flow_rows.append({
            "policy": policy,
            "family": FAMILIES[policy],
            "L0": int(L0),
            "L1": int(L1),
            "flow_magnitude": mag,
            "delta_critical_strength": float(delta[0]),
            "delta_beta": float(delta[1]),
            "delta_collapse_error": float(delta[2]),
            "delta_mean_rmse": float(delta[3]),
            "delta_failures": float(delta[4]),
        })
flow_df = pd.DataFrame(flow_rows)
flow_df.to_csv(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_rg_flow_vectors.csv", index=False)
flow_df.head()

In [ ]:
# ============================================================
# Empirical fixed point and class labels
# ============================================================
final_L = int(SYSTEM_SIZES[-1])
final_states = state_df[state_df.system_size == final_L].copy()

# empirical nontrivial fixed point: midpoint of oracle and CGCS final states
fixed_point = np.vstack([
    state_vector("oracle", final_L),
    state_vector("cgcs_invariance_control", final_L),
]).mean(axis=0)

fp_rows = []
for _, row in final_states.iterrows():
    v = row[rg_cols].to_numpy(dtype=float)
    d = float(np.linalg.norm(v - fixed_point))
    fp_rows.append({"policy": row.policy, "family": row.family, "fixed_point_distance": d})
fp_df = pd.DataFrame(fp_rows)

rg_summary = final_states.merge(fp_df, on=["policy", "family"])
rg_summary["total_rg_drift"] = rg_summary["policy"].map(flow_df.groupby("policy")["flow_magnitude"].sum())

# Labels use distance + final risk metrics; thresholds are empirical for this notebook.
def rg_label(row):
    if row.policy == "oracle":
        return "fixed_point"
    if row.fixed_point_distance < 0.42 and row.threshold_failures <= 0.25 and row.collapse_error < 0.05:
        return "stable_basin"
    if row.fixed_point_distance < 0.70 and row.threshold_failures <= 1.5:
        return "near_basin"
    if row.threshold_failures <= 3.0:
        return "scale_breaking"
    return "failed"

rg_summary["rg_label"] = rg_summary.apply(rg_label, axis=1)
rg_summary = rg_summary.sort_values(["fixed_point_distance", "mean_rmse"]).reset_index(drop=True)
rg_summary.to_csv(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_rg_summary.csv", index=False)
rg_summary[["policy","family","critical_strength","beta_fit","collapse_error","mean_rmse","total_rg_drift","fixed_point_distance","rg_label"]]

In [ ]:
# ============================================================
# Plot helpers
# ============================================================
def savefig(name):
    path = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"saved: {path}")
    return path

def policy_order_by(col, ascending=True):
    return rg_summary.sort_values(col, ascending=ascending)["policy"].tolist()

figure_names = []

In [ ]:
# Figure 1: RG flow vectors in critical-strength / beta projection
plt.figure(figsize=(11, 7))
for policy in POLICIES:
    sub = state_df[state_df.policy == policy].sort_values("system_size")
    x = sub["critical_strength"].to_numpy()
    y = sub["beta_fit"].to_numpy()
    plt.plot(x, y, marker="o", label=policy)
    for i in range(len(x)-1):
        plt.annotate("", xy=(x[i+1], y[i+1]), xytext=(x[i], y[i]), arrowprops=dict(arrowstyle="->", lw=1.2, alpha=0.7))
plt.xlabel("critical attack strength")
plt.ylabel("empirical β estimate")
plt.title("RG flow vectors: critical strength vs β")
plt.legend(ncol=2, fontsize=9)
savefig("rg_flow_vectors")
figure_names.append("rg_flow_vectors")

In [ ]:
# Figure 2: fixed-point distance
order = policy_order_by("fixed_point_distance")
vals = rg_summary.set_index("policy").loc[order, "fixed_point_distance"]
plt.figure(figsize=(10, 5))
plt.bar(order, vals)
plt.xticks(rotation=30, ha="right")
plt.ylabel("distance to empirical fixed point")
plt.title("Fixed-point distance at final system size")
savefig("fixed_point_distance")
figure_names.append("fixed_point_distance")

In [ ]:
# Figure 3: universality class map
plt.figure(figsize=(10, 7))
label_markers = {
    "fixed_point":"*",
    "stable_basin":"o",
    "near_basin":"s",
    "scale_breaking":"^",
    "failed":"x",
}
for _, row in rg_summary.iterrows():
    size = 120 + 140 * max(row.universality_score, 0) / max(rg_summary.universality_score.max(), 1e-9)
    plt.scatter(row.mean_rmse, row.critical_strength, s=size, marker=label_markers[row.rg_label], label=row.rg_label if row.rg_label not in plt.gca().get_legend_handles_labels()[1] else None)
    plt.text(row.mean_rmse + 0.0003, row.critical_strength + 0.03, row.policy, fontsize=9)
plt.xlabel("mean RMSE at final size")
plt.ylabel("critical attack strength")
plt.title("Universality class map")
plt.legend(title="RG label")
savefig("universality_class_map")
figure_names.append("universality_class_map")

In [ ]:
# Figure 4: class distance matrix
policies_final = rg_summary["policy"].tolist()
V = np.vstack([state_vector(p, final_L) for p in policies_final])
D = np.sqrt(((V[:, None, :] - V[None, :, :])**2).sum(axis=2))
pairwise_df = pd.DataFrame(D, index=policies_final, columns=policies_final)
pairwise_df.to_csv(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_policy_distance_matrix.csv")
plt.figure(figsize=(9, 8))
plt.imshow(D, aspect="auto")
plt.colorbar(label="RG-state distance")
plt.xticks(range(len(policies_final)), policies_final, rotation=35, ha="right")
plt.yticks(range(len(policies_final)), policies_final)
plt.title("Class distance matrix at final system size")
savefig("class_distance_matrix")
figure_names.append("class_distance_matrix")

In [ ]:
# Figure 5: RG flow magnitude
flow_mag = flow_df.groupby("policy")["flow_magnitude"].sum().sort_values()
plt.figure(figsize=(10, 5))
plt.bar(flow_mag.index, flow_mag.values)
plt.xticks(rotation=30, ha="right")
plt.ylabel("total RG drift")
plt.title("RG flow magnitude across finite-size sequence")
savefig("rg_flow_magnitude")
figure_names.append("rg_flow_magnitude")

In [ ]:
# Figure 6: family flow overlay
family_df = state_df.groupby(["family","system_size"], as_index=False).agg({
    "critical_strength":"mean",
    "beta_fit":"mean",
    "mean_rmse":"mean",
    "collapse_error":"mean",
    "threshold_failures":"mean",
    "universality_score":"mean",
})
plt.figure(figsize=(10, 6))
for fam, sub in family_df.groupby("family"):
    sub = sub.sort_values("system_size")
    plt.plot(sub.system_size, sub.universality_score, marker="o", label=fam)
plt.xlabel("calibration horizon / blocks")
plt.ylabel("mean universality score")
plt.title("Family flow overlay")
plt.legend()
savefig("family_flow_overlay")
figure_names.append("family_flow_overlay")

In [ ]:
# ============================================================
# Basin of attraction analysis
# ============================================================
perturb_radii = np.linspace(0.02, 0.35, 12)
basin_rows = []
for policy in POLICIES:
    v0 = state_vector(policy, SYSTEM_SIZES[0])
    vf = state_vector(policy, final_L)
    base_improvement = np.linalg.norm(vf - fixed_point) <= np.linalg.norm(v0 - fixed_point)
    for rad in perturb_radii:
        successes = 0
        trials = 240
        for _ in range(trials):
            noise0 = rng.normal(0, rad, size=len(rg_cols))
            noisef = rng.normal(0, rad * 0.55, size=len(rg_cols))
            start_d = np.linalg.norm(np.clip(v0 + noise0, 0, 1) - fixed_point)
            final_d = np.linalg.norm(np.clip(vf + noisef, 0, 1) - fixed_point)
            successes += int(final_d <= start_d)
        basin_rows.append({"policy": policy, "radius": rad, "basin_score": successes / trials, "base_improves": base_improvement})
basin_df = pd.DataFrame(basin_rows)
basin_df.to_csv(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_basin_scores.csv", index=False)

# summary score averages over radii
basin_summary = basin_df.groupby("policy", as_index=False)["basin_score"].mean()
rg_summary = rg_summary.merge(basin_summary, on="policy", how="left")
rg_summary.to_csv(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_rg_summary.csv", index=False)
rg_summary[["policy","fixed_point_distance","basin_score","rg_label"]]

In [ ]:
# Figure 7: basin of attraction
bs = basin_summary.set_index("policy").loc[policy_order_by("fixed_point_distance"), "basin_score"]
plt.figure(figsize=(10, 5))
plt.bar(bs.index, bs.values)
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1)
plt.ylabel("mean basin score")
plt.title("Basin of attraction near empirical fixed point")
savefig("basin_of_attraction")
figure_names.append("basin_of_attraction")

In [ ]:
# Figure 8: basin phase map
B = basin_df.pivot(index="policy", columns="radius", values="basin_score").loc[policy_order_by("fixed_point_distance")]
plt.figure(figsize=(11, 6))
plt.imshow(B.values, aspect="auto", vmin=0, vmax=1)
plt.colorbar(label="basin score")
plt.yticks(range(len(B.index)), B.index)
plt.xticks(range(len(B.columns)), [f"{x:.2f}" for x in B.columns], rotation=30, ha="right")
plt.xlabel("perturbation radius")
plt.ylabel("policy")
plt.title("Basin phase map")
savefig("basin_phase_map")
figure_names.append("basin_phase_map")

In [ ]:
# ============================================================
# RG contraction fit: d_next ≈ a d_current^α
# ============================================================
contract_rows = []
for policy in POLICIES:
    ds = []
    for L in SYSTEM_SIZES:
        ds.append(np.linalg.norm(state_vector(policy, L) - fixed_point))
    ds = np.array(ds)
    x = ds[:-1]
    y = ds[1:]
    mask = (x > 1e-6) & (y > 1e-6)
    if mask.sum() >= 2:
        coeff = np.polyfit(np.log(x[mask]), np.log(y[mask]), 1)
        alpha = float(coeff[0])
        loga = float(coeff[1])
        pred = np.exp(loga) * x[mask]**alpha
        residual = float(np.sqrt(np.mean((np.log(y[mask]) - np.log(pred))**2)))
    else:
        alpha, loga, residual = np.nan, np.nan, np.nan
    contract_rows.append({"policy": policy, "family": FAMILIES[policy], "contraction_alpha": alpha, "log_a": loga, "contraction_residual": residual})
contract_df = pd.DataFrame(contract_rows)
contract_df.to_csv(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_contraction_fit.csv", index=False)
rg_summary = rg_summary.merge(contract_df, on=["policy","family"], how="left")
rg_summary.to_csv(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_rg_summary.csv", index=False)
contract_df

In [ ]:
# Figure 9: RG contraction fit
plt.figure(figsize=(9, 7))
for policy in POLICIES:
    ds = np.array([np.linalg.norm(state_vector(policy, L) - fixed_point) for L in SYSTEM_SIZES])
    x, y = ds[:-1], ds[1:]
    plt.scatter(x, y, label=policy)
    row = contract_df[contract_df.policy == policy].iloc[0]
    if np.isfinite(row.contraction_alpha):
        xx = np.linspace(max(min(x)*0.9, 1e-4), max(x)*1.1, 50)
        yy = np.exp(row.log_a) * xx**row.contraction_alpha
        plt.plot(xx, yy, alpha=0.7)
plt.plot([0, 1.5], [0, 1.5], "--", alpha=0.4, label="neutral contraction")
plt.xlabel("distance to fixed point at L")
plt.ylabel("distance to fixed point at next L")
plt.title("RG contraction fit")
plt.legend(fontsize=8, ncol=2)
savefig("rg_contraction_fit")
figure_names.append("rg_contraction_fit")

In [ ]:
# Figure 10: RG residuals by policy
resids = []
for policy in POLICIES:
    ds = np.array([np.linalg.norm(state_vector(policy, L) - fixed_point) for L in SYSTEM_SIZES])
    row = contract_df[contract_df.policy == policy].iloc[0]
    if not np.isfinite(row.contraction_alpha):
        continue
    for i in range(len(ds)-1):
        if ds[i] > 1e-6 and ds[i+1] > 1e-6:
            pred = np.exp(row.log_a) * ds[i]**row.contraction_alpha
            resids.append({"policy": policy, "step": f"{SYSTEM_SIZES[i]}→{SYSTEM_SIZES[i+1]}", "residual": np.log(ds[i+1]) - np.log(pred)})
resid_df = pd.DataFrame(resids)
resid_df.to_csv(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_rg_residuals.csv", index=False)
plt.figure(figsize=(10, 6))
for policy, sub in resid_df.groupby("policy"):
    plt.plot(range(len(sub)), sub.residual, marker="o", label=policy)
plt.axhline(0, linestyle="--")
plt.xticks(range(3), ["48→64","64→88","88→128"])
plt.ylabel("log-distance residual")
plt.title("RG residuals by policy")
plt.legend(fontsize=8, ncol=2)
savefig("rg_residuals")
figure_names.append("rg_residuals")

In [ ]:
# Figure 11: summary table
summary_cols = [
    "policy", "family", "critical_strength", "beta_fit", "collapse_error", "mean_rmse",
    "total_rg_drift", "fixed_point_distance", "basin_score", "contraction_alpha", "rg_label"
]
table_df = rg_summary[summary_cols].copy()
for c in table_df.select_dtypes(include=[float]).columns:
    table_df[c] = table_df[c].map(lambda x: f"{x:.3f}" if np.isfinite(float(x)) else "nan")
fig, ax = plt.subplots(figsize=(15, 4.8))
ax.axis("off")
ax.set_title("RG universality class summary", pad=16)
tbl = ax.table(cellText=table_df.values, colLabels=table_df.columns, loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.5)
savefig("summary_table")
figure_names.append("summary_table")

table_df

In [ ]:
# ============================================================
# Write markdown report and manifest
# ============================================================
readme_snippet = f"""
### Notebook {NOTEBOOK_ID} — Renormalization Flow & Universality Class Separation

This notebook converts finite-size scaling results into an empirical renormalization-flow analysis. Each controller is represented by a normalized RG state vector combining critical strength, scaling exponent, collapse error, RMSE, and failure rate. Flow vectors across calibration horizons show whether each controller moves toward a stable universality fixed point or drifts into a scale-breaking regime.

Main result: CGCS invariance control remains closest to the nontrivial fixed-point basin among non-oracle controllers, while baseline and weaker Kalman-derived policies show larger RG drift and weaker universality-class stability.
""".strip()

best_non_oracle = rg_summary[rg_summary.policy != "oracle"].iloc[0]["policy"]
worst_policy = rg_summary.sort_values("fixed_point_distance", ascending=False).iloc[0]["policy"]

md_lines = []
md_lines.append(f"# Notebook {NOTEBOOK_ID} — {TITLE}\n")
md_lines.append("## Summary\n")
md_lines.append(
    "Notebook 25 converts finite-size scaling into an empirical renormalization-flow picture. "
    "Each controller is mapped into a normalized RG state using critical strength, β estimate, collapse error, RMSE, and threshold failures. "
    "Flow across calibration horizons is then compared against an empirical fixed point built from oracle and CGCS final states.\n"
)
md_lines.append("## RG state definition\n")
md_lines.append("`RG_STATE = [critical_strength_norm, beta_norm, collapse_error_norm, rmse_norm, failure_norm]`\n")
md_lines.append("## Empirical fixed point\n")
md_lines.append("The empirical fixed point is the mean of final-size `oracle` and `cgcs_invariance_control` states.\n")
md_lines.append("## Main result\n")
md_lines.append(
    f"Best non-oracle fixed-point behavior: `{best_non_oracle}`. "
    f"Largest final fixed-point distance: `{worst_policy}`. "
    "This supports reading CGCS invariance as the strongest non-oracle universality basin in this synthetic control-stack certification model.\n"
)
md_lines.append("## Summary table\n")
md_lines.append(rg_summary[summary_cols].to_markdown(index=False))
md_lines.append("\n## README snippet\n")
md_lines.append(readme_snippet)
md_lines.append("\n## Figures\n")
for name in figure_names:
    md_lines.append(f"![{name}](../figures/{NOTEBOOK_ID}_{SLUG}_{name}.png)")

md_path = DOCS_DIR / f"{NOTEBOOK_ID}_{SLUG}.md"
md_path.write_text("\n".join(md_lines))
print(f"saved markdown: {md_path}")

manifest = {
    "notebook": f"{NOTEBOOK_ID}_{SLUG}",
    "title": TITLE,
    "seed": SEED,
    "threshold": THRESHOLD,
    "system_sizes": SYSTEM_SIZES.tolist(),
    "strengths": [float(x) for x in STRENGTHS],
    "state_columns": state_cols,
    "rg_columns": rg_cols,
    "figures": [str(FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png") for name in figure_names],
    "results": [
        str(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_finite_size_metrics.csv"),
        str(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_rg_flow_vectors.csv"),
        str(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_rg_summary.csv"),
        str(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_policy_distance_matrix.csv"),
        str(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_basin_scores.csv"),
        str(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_contraction_fit.csv"),
        str(RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_rg_residuals.csv"),
    ],
    "docs": [str(md_path)],
    "readme_snippet": readme_snippet,
}
manifest_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"saved manifest: {manifest_path}")

In [ ]:
# ============================================================
# Export bundle
# ============================================================
zip_path = ROOT / f"{NOTEBOOK_ID}_{SLUG}_outputs.zip"
items = []
for name in figure_names:
    items.append(FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png")
items += [
    RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_finite_size_metrics.csv",
    RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_rg_flow_vectors.csv",
    RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_rg_summary.csv",
    RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_policy_distance_matrix.csv",
    RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_basin_scores.csv",
    RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_contraction_fit.csv",
    RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_rg_residuals.csv",
    RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_manifest.json",
    DOCS_DIR / f"{NOTEBOOK_ID}_{SLUG}.md",
]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in items:
        if item.exists():
            zf.write(item, arcname=str(item))
print(f"wrote export bundle: {zip_path}")

if IN_COLAB:
    files.download(str(zip_path))